# GDP and volumes

In this notebook you will learn how to

- compute GDP from the production and the expenditure side
- see what contributes to growth
- get real growth by adding the tables in previous year's prices

## Load the data

See [01_getting_started](01_getting_started.ipynb) for an explanation of these steps.

In [ ]:
from pathlib import Path

import sutlab as sl

data_folder = Path("data")
years = [2016, 2017, 2018, 2019]

metadata = sl.load_metadata_from_excel(
    data_folder / "ta_columns.xlsx",
    data_folder / "ta_classifications.xlsx",
)

current_paths = [data_folder / f"ta_l_{year}.parquet" for year in years]
sut = sl.load_sut_from_separated_parquet(years, current_paths, metadata, "current_year")

## GDP in current prices

`inspect_gdp` computes GDP from both sides. The last row shows the difference between them, which should be close to zero. The data is in thousand DKK. We show it in million DKK.

In [ ]:
gdp = sut.inspect_gdp().set_display_unit(1000).set_display_decimals(1)
gdp.gdp

The growth from year to year, in current prices:

In [ ]:
gdp.gdp_growth

## Contributions to growth

A large component with low growth can matter more than a small one with high growth. The contribution of a component is its change divided by GDP the year before, so contributions are comparable and the components of each side add up to GDP growth.

In [ ]:
gdp.gdp_growth_contributions

## Industries

There is a set of tables with one row per industry. They are long, so here we use the numbers in `.data` and pick the ten industries that contributed the most to growth in 2019.

In [ ]:
contributions = gdp.data.gdp_industries_growth_contributions
total_rows = ["Gross Value Added", "Total product taxes, netto", "GDP"]
is_total = contributions.index.get_level_values(1).isin(total_rows)

industry_contributions = contributions[~is_total]
industry_contributions.sort_values(2019, ascending=False).head(10)

## Volumes

To separate volume from price, we also need the tables in **previous year's prices**: 2019 in 2018 prices, 2018 in 2017 prices, and so on. They are kept in a separate SUT. It uses the same metadata but another price basis.

In [ ]:
previous_paths = [data_folder / f"ta_d_{year}.parquet" for year in years]
sut_previous = sl.load_sut_from_separated_parquet(years, previous_paths, metadata, "previous_year")

Give it to `inspect_gdp` as `previous`.

In [ ]:
gdp = sut.inspect_gdp(previous=sut_previous)
gdp = gdp.set_display_unit(1000).set_display_decimals(1)

The result now has three parts:

- `gdp.gdp`: current prices, as before
- `gdp.previous.gdp`: previous year's prices
- `gdp.real.gdp`: chained volumes

Real growth is in the `real` part.

In [ ]:
gdp.real.gdp_growth

In [ ]:
gdp.real.gdp_growth_contributions

The chained volumes are in the prices of a reference year. Choose it with `reference_id`.

In [ ]:
gdp_2016 = sut.inspect_gdp(previous=sut_previous, reference_id=2016)
gdp_2016 = gdp_2016.set_display_unit(1000).set_display_decimals(0)
gdp_2016.real.gdp

**Note:** chained volumes are not additive. Each row is chained by itself, so components do not in general add up to the total.

## Volumes in the other inspections

`inspect_products`, `inspect_industries` and `inspect_final_uses` also accept `previous`.

In [ ]:
wine = sut.inspect_products("V220400", previous=sut_previous)
wine.real.balance_growth

Compare with growth in current prices:

In [ ]:
wine.balance_growth

---
Next: [04_balancing](04_balancing.ipynb)